# Data Science for Business - Predicting Credit Card Default with KNN

*Session 1 · ISLP 2.2.3 (KNN, eq. 2.12) · 4.4.2 (confusion matrix, threshold, eq. 4.27) · Lab 4.7.6*

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

In [ ]:
pd.options.display.float_format = '{:.2f}'.format  # Turn off scientific notation for large numbers

## Problem description

### Credit card default

Credit card default occurs when a cardholder fails to pay the minimum due on their credit card bill for a certain period of time. This can lead to a number of negative consequences, including late fees, increased interest rates, and damage to the cardholder's credit score. In some cases, credit card default can even result in legal action being taken against the cardholder.

For a bank, it is valuable to predict which customers are likely to default, so that it can act early (e.g., lower the credit limit or contact the customer). We use ISLP's simulated `Default` data set (ISLP Fig. 4.1): 10,000 customers with their monthly credit card `balance`, annual `income`, `student` status, and whether they defaulted (`default`).

## Load data
Importing the dataset from a CSV file.

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_01/default.csv')

In [ ]:
df.head()

## Explore data

### Summary statistics
Generating summary statistics of the numerical variables `balance` and `income` and the class distribution of the `default` variable.

In [ ]:
df.describe()

In [ ]:
df['default'].value_counts()

In [ ]:
df['default'].value_counts(normalize=True)

**The classes are highly imbalanced:** only about 3% of the customers default (333 of 10,000, see ISLP Table 4.4). A trivial model that always predicts "No" already reaches about 97% accuracy. Accuracy alone is therefore a poor yardstick here; we will pay special attention to how many of the actual defaulters ("Yes") the model catches (recall).

### Visualizations
Use seaborn to visually explore the dataset. We create a scatterplot with `balance` and `income` on the x and y axes and `default` as hue (compare ISLP Fig. 4.1).

In [ ]:
sns.scatterplot(x='balance', y='income', hue='default', alpha=0.5, data=df)
plt.xlabel('Balance')
plt.ylabel('Income')
plt.show()

## Prepare data
We first split the data into predictors $X$ (in machine learning often called *features*) and the response $y$ (*labels*).

Then we hold out 20% of the data as a test set that the model never sees during training. This gives us an honest estimate of how well the model will work for new customers (ISLP 2.2.1, 5.1.1). Because defaults are so rare, we use `stratify=y` so that training and test set have the same share of defaulters.

In [ ]:
X = df[['balance', 'income']]
y = df['default']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
y_train.value_counts(normalize=True)

Scale the predictors using `StandardScaler`, as KNN is based on distances and therefore sensitive to the scale of the variables (`income` is measured in much larger numbers than `balance`).

The scaler learns mean and standard deviation from the **training data only** and then applies the same transformation to the test data; otherwise information from the test set would leak into training. (Lab 4.7.6 in ISLP fits the scaler on all data before splitting; we deliberately do it the cleaner way.)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## K-nearest neighbors (KNN)

### Train the model
Fit the KNN model to the training data using $K=5$ (ISLP 2.2.3).

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

### Predict probabilities
Use the fitted model to make predictions on the test data. More specifically, we calculate the probabilities of the positive class (default = "Yes", the second column of `predict_proba`).

KNN's predicted probability is simply the share of defaulters among the $K$ nearest neighbours $\mathcal{N}_0$ of a test observation $x_0$ (ISLP eq. 2.12):

$$\Pr(Y=j \mid X=x_0)=\frac{1}{K}\sum_{i\in\mathcal{N}_0} I(y_i=j)$$

With $K=5$, only the values 0, 0.2, 0.4, 0.6, 0.8 and 1 are possible.

In [ ]:
knn.classes_

In [ ]:
y_prob = knn.predict_proba(X_test_scaled)[:, 1]

In [ ]:
y_prob

Let's plot the distribution of predicted probabilities for default.

In [ ]:
sns.histplot(y_prob)
plt.xlabel('Probability of Default')
plt.ylabel('Frequency')
plt.show()

### Classify with a decision threshold
Transform probabilities into binary outcomes using a threshold of 0.5, i.e., predict "Yes" if $\Pr(\text{default}=\text{Yes}\mid X=x) > 0.5$ (ISLP eq. 4.26). With $K=5$, this means that at least 3 of the 5 neighbours defaulted.

In [ ]:
decision_threshold = 0.5
y_pred = np.where(y_prob > decision_threshold, 'Yes', 'No')

### Evaluate the model
Plot a confusion matrix to evaluate the model's performance.

**Attention:** scikit-learn shows the **true classes in the rows** and the **predicted classes in the columns**. ISLP's confusion matrices (e.g., Table 4.4) are the other way round (predicted in the rows, truth in the columns). Always check the axis labels!

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=['No', 'Yes'], cmap='Blues')
plt.show()

The bottom-left cell contains the defaulters that we missed (false negatives). For a bank, this is usually the costliest error. The top-right cell contains customers who were wrongly flagged as defaulters (false positives).

For comparison, here is the same table in the layout used by ISLP (predicted in the rows, truth in the columns):

In [ ]:
pd.crosstab(y_pred, y_test, rownames=['Predicted'], colnames=['Truth'], margins=True)

Print a classification report with precision, recall, F1-score, and accuracy.
* The recall of class "Yes" is the share of actual defaulters that the model detects. Compare it with the overall accuracy!
* The macro average F1 score is computed using the arithmetic mean of all the per-class F1 scores. That is, all classes are treated equally regardless of their support values.
* The weighted average F1 score is computed by averaging the F1 scores of each class, weighted by their support values.

In [ ]:
print(classification_report(y_test, y_pred))

## Your turn!
**Your task:** Experiment with the above code and:

1. Change `n_neighbors` (e.g., 1, 10, 50) and observe how accuracy **and** recall for "Yes" change.
2. Lower the `decision_threshold`, e.g., to 0.2 as in ISLP eq. 4.27 and Table 4.5. How do the false negatives and false positives change? Which threshold would you recommend to the bank, and why is accuracy not a good guide for this decision?

In [ ]:
# YOUR CODE HERE